[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nube1/h-potential-companions/blob/main/methods-series/notebooks/02_prognostic.ipynb)

# Paper 2: attenuation, and what measuring instead of observing costs

Fit a survival model on a factor score rather than on the construct itself and the
coefficient shrinks toward the null. That shrinkage has a size, and the size has a
consequence for how many events a study needs. The consequence has a floor no instrument
can beat.

## Status, before anything else

- **Every datum here is synthetic.** Nothing in this notebook was measured on a person.
- **Papers 1 to 3 are protocols.** They say what will be done, not what was found. No
  participant has been recruited and no Delphi panel has met.
- **Papers 4 to 6 are method illustrations.** Their inputs show a workflow. They are not
  estimates from any health system and not claims about any product.
- **No figure below is evidence about patients.** Each number is a property of a stated
  model or a stated decision rule, and nothing more.

## Setup

The cell below installs the package straight from GitHub when this runs in Colab, and installs the local checkout when it does not. It prints the resolved version so you know what you are running.

**Runtime.** Reduced run: about 40 seconds. Full run: about ninety seconds.

In [ ]:
# ---------------------------------------------------------------- setup
# Repository holding this package. The repository is private: in Colab, sign
# in to GitHub or clone it yourself before running.
GITHUB_OWNER = "Nube1"
REPO = "h-potential-companions"
SUBDIR = "methods-series"

import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q "hpotential[full] @ git+https://github.com/{GITHUB_OWNER}/{REPO}.git#subdirectory={SUBDIR}"
else:
    # Running from a clone: install the checkout this notebook sits in.
    import importlib.util
    import pathlib
    import subprocess

    if importlib.util.find_spec("hpotential") is None:
        for parent in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
            if (parent / "src" / "hpotential").exists():
                subprocess.run(
                    [sys.executable, "-m", "pip", "install", "-q", "-e", str(parent)], check=True
                )
                break
        else:
            raise RuntimeError(
                "hpotential is not importable and no local checkout was found. "
                "Run this in Colab with access to the repository, or clone it."
            )

import hpotential

print(f"hpotential {hpotential.__version__}")
print(f"running in Colab: {IN_COLAB}")

In [ ]:
# Importing hpotential selects the Agg backend, because the library writes figures to
# files. Put the plots back into the page, then borrow the series figure style.
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hpotential.utils.plotting import GRID, INK, apply_style, series_color

apply_style()
plt.rcParams["figure.dpi"] = 110  # 200 is for print; this is for a screen

try:
    import ipywidgets as widgets
    from IPython.display import display

    HAVE_WIDGETS = True
except Exception:  # Colab sometimes ships without a working widget stack
    HAVE_WIDGETS = False


def explore(fn, controls, defaults):
    """Draw the static case first, then attach sliders if ipywidgets works.

    The static call means the notebook still teaches something when it is read on
    GitHub, where no widget ever runs.
    """
    fn(**defaults)
    if HAVE_WIDGETS:
        display(widgets.interactive(fn, **controls))
    else:
        print("ipywidgets is unavailable here, so the static case above is all there is.")


print("plots inline, style applied, widgets:", HAVE_WIDGETS)

In [ ]:
# ---------------------------------------------------------------- run size
FULL_RUN = False   # True reproduces the published values. Full run here takes about ninety seconds.


def banner():
    """Print at the top of every results cell. It has to be impossible to miss."""
    if FULL_RUN:
        print("=" * 74)
        print("FULL RUN: published replicate counts. These are the published values.")
        print("=" * 74)
    else:
        print("=" * 74)
        print("REDUCED RUN: cut replicate counts for speed.")
        print("The numbers below are NOT the published values. Set FULL_RUN = True.")
        print("=" * 74)


banner()

## The claim

Let $\zeta$ be the standardised construct and $\widehat\zeta$ its regression score, with
determinacy $\rho_D$. Under a proportional-hazards model with hazard
$h(t) \exp(\beta\zeta)$ and measurement error that carries no information about survival,
the coefficient fitted on the standardised score is, to first order,

$$\beta_{\text{obs}} \approx \rho_D\,\beta_{\text{true}} .$$

This is classical regression dilution with the reliability replaced by the determinacy.
The step worth care is that $\rho_D$, not $\rho_D^2$, appears: the score is standardised,
so the shrinkage is the correlation.

In [ ]:
banner()

from hpotential.analyses.paper2_prognostic import population_model
from hpotential.config import Paper2Config
from hpotential.core.measurement import determinacy_ceiling
from hpotential.core.prognostic import (
    attenuated_beta, events_inflation, pipeline_replicate, required_events,
)

cfg = Paper2Config()
model = population_model(cfg)
rho_planned = model.determinacy()
rho_inf, c = determinacy_ceiling(np.asarray(cfg.gamma), np.asarray(cfg.psi))

print(f"planned design: {cfg.per_domain} indicators per domain, standardised loading {cfg.lam_std}")
print(f"rho_D of the planned design   {rho_planned:.4f}")
print(f"ceiling for this structure    {rho_inf:.4f}")
print()
print(f"beta_true                     {cfg.beta_true:+.4f}   (HR {np.exp(cfg.beta_true):.4f} per SD)")
print(f"predicted attenuated beta     {attenuated_beta(cfg.beta_true, rho_planned):+.4f}"
      f"   (HR {np.exp(attenuated_beta(cfg.beta_true, rho_planned)):.4f})")

## Simulate the whole pipeline

A prediction about attenuation is worth having only if it survives the pipeline it
describes. One replicate does all of it: draw people, draw their indicators from the
measurement model, fit the factor model on those indicators, score it, simulate survival
from the true construct, then fit two Cox models, one on the true construct and one on the
score. The ratio of the two coefficients is the attenuation, and both come from the same
people, so the ratio is not polluted by differences in the survival data.

In [ ]:
banner()

import time

reps = cfg.reps if FULL_RUN else 20
n_grid = cfg.n_grid if FULL_RUN else (1500, 2500)

start = time.time()
records = []
for endpoint_index, (endpoint, fraction) in enumerate(cfg.event_fractions):
    for n in n_grid:
        for rep in range(reps):
            rng = np.random.default_rng([cfg.seed, endpoint_index, int(n), int(rep)])
            rec = pipeline_replicate(int(n), endpoint, fraction, model, cfg.beta_true, rng,
                                     per_domain=cfg.per_domain)
            if rec is not None:
                records.append(rec)
print(f"{len(records)} replicates in {time.time() - start:.1f} s")

frame = pd.DataFrame([{"endpoint": r.endpoint, "n": r.n, "events": r.events,
                       "beta_oracle": r.beta_oracle, "beta_observed": r.beta_observed,
                       "rho_hat": r.rho_hat, "significant": r.significant,
                       "significant_oracle": r.significant_oracle} for r in records])

grouped = frame.groupby(["endpoint", "n"]).agg(
    reps=("events", "size"), events=("events", "mean"),
    beta_oracle=("beta_oracle", "mean"), beta_observed=("beta_observed", "mean"),
    power=("significant", "mean"), power_oracle=("significant_oracle", "mean"),
).reset_index()
grouped["attenuation"] = grouped["beta_observed"] / grouped["beta_oracle"]
grouped.round(4)

In [ ]:
banner()

pooled = frame["beta_observed"].mean() / frame["beta_oracle"].mean()
print(f"pooled attenuation ratio over {len(frame)} replicates   {pooled:.4f}")
print(f"predicted by rho_D of the population model              {rho_planned:.4f}")
print(f"difference                                              {abs(pooled - rho_planned):.4f}")
print()
print("Pooling matters. A ratio from one design point carries several per cent of Monte")
print("Carlo error, which is larger than the thing being estimated, so the per-point")
print("ratios scatter around the prediction by more than the quantity itself.")
print(f"per-point range in this run: {grouped['attenuation'].min():.3f} to "
      f"{grouped['attenuation'].max():.3f}")

In [ ]:
banner()

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))

ax = axes[0]
for i, endpoint in enumerate(grouped["endpoint"].unique()):
    part = grouped[grouped["endpoint"] == endpoint].sort_values("n")
    ax.plot(part["n"], part["attenuation"], "o-", ms=5, lw=1.4, color=series_color(i),
            label=endpoint)
ax.axhline(rho_planned, ls="--", lw=1.1, color=INK)
ax.text(grouped["n"].max(), rho_planned + 0.006, rf"predicted $\rho_D$ = {rho_planned:.4f}",
        ha="right", fontsize=8)
ax.set_xlabel("sample size $n$"); ax.set_ylabel("attenuation ratio")
ax.set_title("Fitted shrinkage against the prediction", fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
width = 0.35
labels = [f"{e}\nn={n}" for e, n in zip(grouped["endpoint"], grouped["n"])]
pos = np.arange(len(grouped))
ax.bar(pos - width / 2, grouped["power_oracle"], width, color=series_color(0), label="oracle")
ax.bar(pos + width / 2, grouped["power"], width, color=series_color(1), label="on the score")
ax.set_xticks(pos); ax.set_xticklabels(labels, fontsize=7)
ax.set_ylabel("power"); ax.set_ylim(0, 1.05)
ax.set_title("The same study, two covariates", fontsize=10)
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## The power corollary

For a single standardised covariate the information about $\beta$ from $d$ events is about
$d$, so the noncentrality of the score test is about $\beta^2 d$. Matching noncentrality
between the oracle study and the study that uses the score requires

$$\beta_{\text{obs}}^2\, d_{\text{score}} = \beta_{\text{true}}^2\, d_{\text{oracle}},$$

and substituting $\beta_{\text{obs}} = \rho_D\beta_{\text{true}}$ gives

$$\frac{d_{\text{score}}}{d_{\text{oracle}}} = \frac{1}{\rho_D^{2}} > \frac{1+c}{c} .$$

The lower bound depends only on the second-order structure. No number of indicators and no
number of participants brings the ratio below it.

In [ ]:
banner()

target_power, alpha = cfg.power, cfg.alpha
rows = []
for label, rho in [("construct observed (oracle)", 1.0),
                   ("at the structural ceiling", rho_inf),
                   ("at the planned design", rho_planned)]:
    rows.append({"design": label, "rho_D": rho,
                 "events inflation": events_inflation(rho),
                 "events for 90% power": required_events(cfg.beta_true, target_power, alpha, rho)})
events = pd.DataFrame(rows)
events["events for 90% power"] = np.ceil(events["events for 90% power"])
events.round(4)

In [ ]:
banner()

oracle_d, ceiling_d, planned_d = np.ceil(
    [required_events(cfg.beta_true, target_power, alpha, r) for r in (1.0, rho_inf, rho_planned)]
)
print(f"detecting HR {np.exp(cfg.beta_true):.3f} per SD at 90% power, 5% two-sided:")
print(f"  {oracle_d:.0f} events if the construct is observed")
print(f"  {ceiling_d:.0f} events at the ceiling      (+{100 * (ceiling_d / oracle_d - 1):.1f}%)")
print(f"  {planned_d:.0f} events at the planned design (+{100 * (planned_d / oracle_d - 1):.1f}%)")
print()
print(f"The {ceiling_d:.0f} is a floor. Every instrument built on gamma = {tuple(cfg.gamma)}")
print(f"and Psi = {tuple(cfg.psi)} pays it, however many items it has, because")
print("rho_D approaches the ceiling from below and never passes it.")
print()
print("A 30 per cent increase in events is a design cost, not a barrier. What the")
print("corollary rules out is the belief that the cost can be written away in items.")

## Interactive: move $\rho_D$

One slider. It sets the determinacy and everything else follows: the attenuated hazard
ratio, the inflation factor, and the events needed for 90 per cent power. The dotted line
marks the ceiling for this second-order structure. Everything to the right of it is
unreachable.

In [ ]:
banner()


def power_panel(rho_D=0.8361):
    beta_obs = attenuated_beta(cfg.beta_true, rho_D)
    d_needed = required_events(cfg.beta_true, target_power, alpha, rho_D)

    grid = np.linspace(0.50, 1.0, 300)
    d_curve = [required_events(cfg.beta_true, target_power, alpha, r) for r in grid]

    fig, ax = plt.subplots(figsize=(6.6, 3.8))
    ax.plot(grid, d_curve, lw=1.6, color=series_color(0))
    ax.axvline(rho_inf, ls=":", lw=1.4, color=series_color(5))
    ax.axvline(rho_D, ls="-", lw=1.0, color=series_color(1))
    ax.plot([rho_D], [d_needed], "o", ms=7, color=series_color(1))
    ax.fill_betweenx([0, max(d_curve)], rho_inf, 1.0, color=GRID, alpha=0.6)
    ax.text(rho_inf - 0.005, max(d_curve) * 0.9, "unreachable", ha="right", fontsize=8,
            color=series_color(5))
    ax.set_ylim(0, min(max(d_curve), 3000))
    ax.set_xlabel(r"determinacy $\rho_D$")
    ax.set_ylabel("events for 90% power")
    ax.set_title(f"rho_D = {rho_D:.4f}: {np.ceil(d_needed):.0f} events", fontsize=10)
    plt.show()

    print(f"attenuated log HR   {beta_obs:+.4f}")
    print(f"attenuated HR       {np.exp(beta_obs):.4f}   against a true {np.exp(cfg.beta_true):.4f}")
    print(f"events inflation    {events_inflation(rho_D):.4f}")
    print(f"events for 90% power {np.ceil(d_needed):.0f}   against {oracle_d:.0f} for an oracle study")
    if rho_D > rho_inf:
        print()
        print(f"NOTE: {rho_D:.4f} is above the ceiling {rho_inf:.4f}. No instrument")
        print("built on this second-order structure can reach it.")


explore(power_panel, controls=dict(rho_D=(0.50, 0.99, 0.005)), defaults=dict(rho_D=0.8361))

---

## What this does not establish

**Attenuation here is a first-order result.** It assumes joint normality, a standardised
covariate, and measurement error that carries no information about survival given
covariates. Under a frailty drift the observed coefficient moves further toward the null
over follow-up, so the true inflation is if anything larger than $1/\rho_D^2$. The bound
is conservative in the direction that matters.

**The events calculation is the standard normal approximation** for a single standardised
covariate, with information about $\beta$ from $d$ events taken as $d$. Adjust for other
covariates and the constant changes; the direction and the bound do not.

**The survival data are simulated from the model.** No cohort has been followed, no event
has been observed, and the event fractions are design assumptions.

**"947 events is a floor" is a statement about this second-order structure.** A different
instrument with genuinely larger $\gamma$ has a different, lower floor. The claim is that
you cannot get under the floor by adding items to this one.

**A reduced run does not reproduce the published attenuation ratio.** The events arithmetic
is closed form and is the same under either flag.